# Reactive machine agents with Mesa (v3.x)

In this notebook you will:
- install **Mesa 3.x** with visualisation support,
- implement a **reactive agent** that monitors the temperature of a machine,
- build a simple **factory model** with several machine agents,
- start an **interactive visualisation** directly in the notebook and explore different parameters,
- extend the model with **maintenance** and **order agents**.


## 1. Installation
First we install Mesa **3.0.3** including visualisation support.


In [ ]:
#!pip install -U mesa[viz]==3.0.3 altair==5.2.0 networkx solara

## 2. Mesa basics
Mesa models consist of three main parts:
- a **Model** class that manages the global state,
- one or more **Agent** classes that define the behaviour of individual agents,
- an optional **visualisation** to observe the model while it runs.

**Important changes in Mesa 3.x compared with v2.x:**
- `Agent.__init__` no longer needs `unique_id` – it is assigned automatically.
- The scheduler (`RandomActivation`) no longer exists → use `self.agents.shuffle_do("step")` instead.
- `model.schedule.agents` → `model.agents`.
- Removing agents: `self.remove()` instead of `self.model.schedule.remove(self)`.


## 3. Implementing the `MachineAgent`
The `MachineAgent` represents a single machine in a factory.
It has a temperature and a simple reactive rule.


In [ ]:
from mesa import Agent
import random


class MachineAgent(Agent):
    """Reactive agent that represents a machine and monitors its temperature.

    Rule:
        temperature > threshold            -> state = 'HOT'
        temperature > 0.75 * threshold     -> state = 'WARM'
        temperature < 20                   -> state = 'COOL'
        otherwise                          -> state = 'OK'
    """

    def __init__(self, model, threshold=70):
        super().__init__(model)          # Mesa 3.x: no unique_id argument
        self.temperature = 20.0
        self.threshold = threshold
        self.state = "OK"
        self.busy = False                # used by the OrderAgent

    def sense_temperature(self):
        """Simple sensor model: temperature + random noise."""
        if self.state != "HOT":
            noise = random.uniform(-1, 2)
            self.temperature += noise

    def decide(self):
        """Reactive decision rule based only on the current temperature."""
        if self.temperature > self.threshold:
            self.state = "HOT"
        else:
            self.state = "OK"

    def act(self):
        """Cool the machine down when it is overheated."""
        if self.state == "HOT":
            self.temperature -= 10

    def step(self):
        """Agent step = sense → decide → act."""
        self.sense_temperature()
        self.decide()
        self.act()


## 4. Implementing the `FactoryModel`
The model places several machines on a grid and activates them in random order. A **DataCollector** tracks how many machines are in the `"HOT"` state.


In [ ]:
from mesa import Model
from mesa.space import MultiGrid
from mesa.datacollection import DataCollector


def count_hot_machines(model):
    return sum(1 for a in model.agents if isinstance(a, MachineAgent) and a.state == "HOT")

# Reference to the model that is currently displayed (used by post_process)
CURRENT_MODEL = {}


class FactoryModel(Model):
    """Simple factory model with a grid of machine agents."""

    def __init__(self, width=10, height=10, density=0.3, threshold=70, seed=None):
        super().__init__(seed=seed)
        CURRENT_MODEL["model"] = self  # register the model that is currently displayed
        self.width = width
        self.height = height
        self.density = density
        self.threshold = threshold

        self.grid = MultiGrid(width, height, torus=False)

        for x in range(self.width):
            for y in range(self.height):
                if self.random.random() < self.density:
                    agent = MachineAgent(self, threshold=self.threshold)
                    self.grid.place_agent(agent, (x, y))

        self.datacollector = DataCollector(
            model_reporters={
                "HotMachines":  count_hot_machines,
            }
        )
        self.datacollector.collect(self)

    def step(self):
        self.agents.shuffle_do("step")
        self.datacollector.collect(self)


## 5. Visualisation with Mesa 3.x (SolaraViz)
Mesa 3.x uses **SolaraViz** as its visualisation framework.

The key is the `post_process` function, which is called after the agents have been drawn and allows arbitrary Matplotlib customisation – here we use it to write the temperature as text into each cell.

⚠️ **Note:** If the display does not work directly in the notebook,
save the code to a `.py` file and start it with `solara run file.py`.


In [ ]:
from mesa.visualization import SolaraViz, make_plot_component, make_space_component


# ── Agent portrayal ─────────────────────────────────────────────────────────
def agent_portrayal(agent):
    color = {
        "HOT":  "red",
        "OK":   "green",
    }.get(agent.state, "gray")

    return {
        "color":  color,
        "size":   600,
        "marker": "s",
        "alpha":  0.9,
    }


# ── Post-processing: temperature text and grid ───────────────────────────────────
def post_process(ax):
    ax.figure.set_size_inches(6, 6)

    legend = ax.get_legend()
    if legend:
        legend.remove()

    for agent in CURRENT_MODEL["model"].agents:
        if not isinstance(agent, MachineAgent):
            continue
        x, y = agent.pos
        ax.text(
            x, y,
            f"{agent.temperature:.1f}°C",
            ha="center", va="center",
            fontsize=7, color="white", fontweight="bold",
        )

    ax.grid(True, alpha=0.3)


# ── Components ────────────────────────────────────────────────────────────────
space_component = make_space_component(
    agent_portrayal,
    post_process=post_process,
)

plot_component = make_plot_component(
    {"HotMachines": "red"}
)

# ── Model parameters with sliders ─────────────────────────────────────────
model_params = {
    "width":  10,
    "height": 10,
    "density": {
        "type":  "SliderFloat",
        "value": 0.3,
        "label": "Density",
        "min":   0.1,
        "max":   1.0,
        "step":  0.1,
    },
    "threshold": {
        "type":  "SliderInt",
        "value": 70,
        "label": "Temperature threshold",
        "min":   30,
        "max":   100,
        "step":  5,
    },
}

# ── Model instance and visualisation ───────────────────────────────────────────
model_instance = FactoryModel(width=10, height=10, density=0.3, threshold=70)

page = SolaraViz(
    model_instance,
    components=[space_component, plot_component],
    model_params=model_params,
    name="Reactive machine agents",
)
page


## 6. Testing the simulation without visualisation
If you only want to check the model logic, you can also run the model directly for a few steps.


In [ ]:
model = FactoryModel(width=10, height=10, density=0.3, threshold=70, seed=42)
for _ in range(20):
    model.step()

model.datacollector.get_model_vars_dataframe().tail()


## 7. Exploration tasks (for the lab)
Use the running model to explore the behaviour of the reactive agents:

1. **Change the threshold**
   - Start with `threshold=70` and then try 60 or 80.
   - What happens to the number of `"HOT"` machines over time?

2. **Change the density**
   - Increase or decrease the `density` parameter.
   - How does this influence the overall state of the system?

3. **Extend the state logic (optional)**
   - Add further states, e.g. `"CRITICAL"`, and map them to new colours in `agent_portrayal`.
   - Define your own temperature ranges for each state.

4. **(Advanced) Add maintenance agents**
   - Create a second agent type that moves through the grid and "repairs" overheated
     machines by resetting their temperature and state.

5. **(Advanced) Add order agents**
   - Create a third agent type that moves orders through the grid and
     looks for free machines by itself.
